# 🧬 Vector Embeddings en Databricks — con la API de OpenAI y con SQL

## ¿Qué vamos a hacer en este notebook?

En este notebook vamos a aprender a **convertir texto en números** usando los **modelos de embeddings** que ya vienen alojados en Databricks. Lo haremos de **dos formas distintas** (las dos sirven, según el contexto):

1. **Con la API de OpenAI** (desde código Python) → ideal para aplicaciones.
2. **Con SQL** usando la función `ai_query` → ideal para procesar tablas enteras dentro del *lakehouse*.

> 🎯 **¿Por qué es tan importante este notebook?** Los *embeddings* son **el corazón de RAG** (*Retrieval-Augmented Generation*). Sin embeddings no hay búsqueda semántica, y sin búsqueda semántica no hay RAG. Todo lo que veremos en los siguientes notebooks (crear el índice vectorial, recuperar documentos, montar la aplicación) se apoya en lo que aprendas aquí.

---

### 🗺️ ¿Dónde encaja esto dentro del flujo de RAG?

RAG consiste en darle a un LLM **información extra y actualizada** en el momento de responder, buscándola primero en *tus* documentos. El flujo completo del curso es:

| Paso | Notebook | Qué hace |
|------|----------|----------|
| 1 | `Data_Prep` | Trocea (*chunking*) los documentos en fragmentos. |
| **2** | **`Vector_Embeddings` (este)** | **Convierte cada fragmento de texto en un vector numérico.** |
| 3 | `Vector_Index_Creation` | Guarda esos vectores en un índice para poder buscarlos. |
| 4 | `RAG_Application` | Busca los fragmentos relevantes y se los pasa al LLM. |
| 5 | `RAG_Evaluation` | Mide si las respuestas son buenas. |

> 📚 **Nota para el aprendizaje**: a lo largo del notebook verás celdas de texto (como esta) que explican el *qué* y el *por qué*, y celdas de código que ejecutan acciones. Lee siempre el texto antes de ejecutar el código de debajo.

## 🧠 Conceptos clave: ¿qué es un *embedding*?

Un **embedding** (o *incrustación vectorial*) es una **lista de números** (un *vector*) que representa el **significado** de un texto. Un modelo de embeddings lee una frase y la transforma en, por ejemplo, 1024 números.

La magia está en que **textos con significados parecidos producen vectores parecidos** (quedan "cerca" en el espacio), aunque usen palabras distintas:

```
"el gato duerme en el sofá"   → [0.12, -0.04, 0.88, ...]  ┐ cerca
"un felino descansa en el sillón" → [0.11, -0.02, 0.85, ...] ┘ (significan casi lo mismo)

"tipos de interés del banco central" → [-0.7, 0.3, 0.05, ...]  ← lejos (otro tema)
```

### 📖 Vocabulario esencial

| Concepto | ¿Qué es? |
|----------|----------|
| **Embedding / Vector** | La representación numérica del *significado* de un texto. |
| **Dimensión** | Cuántos números tiene el vector (p. ej. `databricks-gte-large-en` → **1024**). Más dimensiones ≈ más matices. |
| **Espacio vectorial** | El "mapa" imaginario donde viven todos los vectores. Lo semánticamente parecido cae junto. |
| **Búsqueda semántica** | Buscar por *significado* en vez de por palabras exactas. Es lo que hace RAG. |
| **Similitud del coseno** (*cosine similarity*) | La forma más común de medir cuán "cerca" están dos vectores. Va de **-1** (opuestos) a **1** (idénticos en significado). |
| **Modelo de embeddings** | El modelo de IA que convierte texto → vector. Aquí usamos `databricks-gte-large-en`. |

### 🔑 La idea que lo conecta todo con RAG

> Cuando un usuario hace una pregunta, también la convertimos en un vector y buscamos los **fragmentos de documento cuyo vector esté más cerca** del vector de la pregunta. Esos fragmentos son la "información relevante" que le damos al LLM para que responda con datos reales. **Eso es la "R" (Retrieval) de RAG.**

### 🤝 ¿Por qué usamos el cliente de OpenAI para hablar con Databricks?

Databricks expone sus modelos (incluidos los de embeddings) con el **mismo protocolo/formato que OpenAI**. Es un estándar de la industria, así que podemos reutilizar la librería `openai` aunque el modelo sea de Databricks. Solo cambiamos a qué `base_url` apunta.

## 1️⃣ Instalación de librerías y utilidades

Necesitamos la librería **`openai`** de Python (fijamos la versión `1.69.0` con `==` para que el notebook sea **reproducible** y no se rompa con futuras versiones).

### 🔧 `!pip` vs `%pip`

| Comando | Qué hace | Recomendación |
|---------|----------|---------------|
| `!pip install ...` | Ejecuta `pip` como comando de shell. Solo instala en el nodo *driver*. | Funciona en notebooks sencillos. |
| `%pip install ...` | *Magic command* de Databricks: instala en **todos los nodos** del cluster y prepara el reinicio del kernel. | ✅ **Preferido en Databricks.** |

> 💡 **Para la certificación**: en Databricks lo recomendado es `%pip`. Tras instalar librerías, conviene reiniciar el intérprete con `dbutils.library.restartPython()` para que el notebook "vea" las versiones nuevas.

In [ ]:
!pip install openai==1.69.0

## 2️⃣ Creando el cliente de OpenAI (apuntando a Databricks)

Un **cliente** es un objeto de Python que sabe comunicarse con un servidor. En vez de escribir tú las peticiones HTTP a mano, te ofrece métodos cómodos como `client.embeddings.create(...)`.

Para crearlo necesitamos dos datos:

| Parámetro | Qué es | Dónde se consigue |
|-----------|--------|-------------------|
| `api_key` | Tu **token de acceso personal (PAT)**. Funciona como una contraseña que identifica quién hace la petición. | En Databricks: *Settings → Developer → Access Tokens → Generate new token*. |
| `base_url` | La **dirección** del servicio de *serving* de tu workspace. Termina en `/serving-endpoints`. | `https://<tu-workspace>.cloud.databricks.com/serving-endpoints` |

### 🔐 MUY IMPORTANTE: no escribas tokens en el código

Aquí el token está escrito directamente (`"YOUR_DATABRICKS_ACCESS_TOKEN"`) **solo por simplicidad didáctica**. En un proyecto real **NUNCA** se escribe una contraseña en el código, porque quedaría visible para todo el que vea el notebook o el repositorio de Git.

La forma correcta y segura es usar **Databricks Secrets**:

```python
api_key = dbutils.secrets.get(scope="mi-scope", key="databricks-token")
```

O, dentro de un notebook de Databricks, obtener el host y un token automático del propio contexto:

```python
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host  = "https://" + ctx.tags().get("browserHostName").get()
token = ctx.apiToken().get()
```

> 💡 **Para la certificación**: los *secret scopes* son la forma recomendada de gestionar credenciales (tokens, claves de API). Es un punto típico de examen.

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key = "YOUR_DATABRICKS_ACCESS_TOKEN",
    base_url = "YOUR_DATABRICKS_WORKSPACE_HOSTNAME/serving-endpoints"
)

## 3️⃣ Generando un embedding con la API de OpenAI

Aquí hacemos nuestra **primera conversión de texto → vector**. Fíjate en que usamos `client.embeddings.create(...)` (no `chat.completions`, porque **no** queremos una respuesta conversacional, sino el vector que representa el texto).

### 🎛️ Parámetros de la petición

| Parámetro | Qué es |
|-----------|--------|
| `model` | El **endpoint del modelo de embeddings** en Databricks. Aquí `databricks-gte-large-en` (modelo *GTE Large* para inglés). |
| `input` | El **texto** que queremos convertir en vector. Puede ser una cadena o una **lista** de cadenas (para procesar muchas a la vez, más eficiente). |

### 📦 Cómo leer la respuesta

El objeto que devuelve tiene esta forma (simplificada):

```
completion.data        → lista de resultados (uno por cada texto de entrada)
completion.data[0].embedding → el vector (lista de floats) del primer texto
```

Por eso el código hace:
- `len(completion.data[0].embedding)` → cuántos números tiene el vector. Para `databricks-gte-large-en` será **1024** (su número de dimensiones).
- `completion.data[0].embedding` → el vector completo.

> 🧩 **Foundation Model APIs**: modelos preinstalados como `databricks-gte-large-en` se facturan por *pay-per-token* (pagas por uso) y no necesitas desplegar nada. Son los *Foundation Model APIs* de Databricks.

> ⚠️ **Consistencia obligatoria**: el **mismo modelo** que uses para indexar tus documentos debe usarse para convertir la pregunta del usuario. Vectores de modelos distintos **no son comparables** entre sí.

In [ ]:
# Petición de embedding a través de la API de OpenAI (apuntando a Databricks)
completion = client.embeddings.create(
    model="databricks-gte-large-en",   # modelo de embeddings (1024 dimensiones)
    input="hi how are you?"            # texto que convertiremos en vector
)

# Número de dimensiones del vector (cuántos números lo componen)
print("length \n")
print(len(completion.data[0].embedding))

# El vector completo: la representación numérica del significado del texto
print("\n embeddings: \n")
print(completion.data[0].embedding)


### 🔬 Experimento: ¿de verdad los vectores "entienden" el significado?

Acabamos de ver un vector, pero un montón de números no dice mucho a simple vista. Vamos a **comprobar la magia** con un mini-experimento que es la esencia de RAG:

1. Convertimos en vectores **4 frases**: una pregunta y tres frases candidatas.
2. Medimos la **similitud del coseno** entre la pregunta y cada candidata.
3. Veremos que la frase con **significado parecido** gana, **aunque use otras palabras**.

#### 📐 ¿Qué es la similitud del coseno?

Mide el **ángulo** entre dos vectores (no su tamaño). Intuición:

| Valor | Significado |
|-------|-------------|
| **≈ 1.0** | Apuntan casi en la misma dirección → **significados muy parecidos**. |
| **≈ 0.0** | Perpendiculares → **sin relación**. |
| **< 0** | Direcciones opuestas → significados contrarios. |

> 🧮 Fórmula: `cos(A, B) = (A · B) / (‖A‖ · ‖B‖)` — el producto escalar dividido por el producto de las longitudes. La usamos en vez de la distancia normal porque **solo nos importa la dirección** (el "tema"), no la magnitud.

In [ ]:
import math

# 1) La pregunta del usuario y tres frases candidatas
question = "How do I reset my account password?"
candidates = [
    "Steps to recover and change your login credentials",  # mismo significado, otras palabras
    "The restaurant serves pasta on Friday nights",         # tema totalmente distinto
    "Click 'Forgot password' to set a new one",             # también relacionado
]

# 2) Convertimos TODO en vectores en una sola llamada (pasamos una lista en 'input')
resp = client.embeddings.create(
    model="databricks-gte-large-en",
    input=[question] + candidates
)
vectors = [item.embedding for item in resp.data]
q_vec, cand_vecs = vectors[0], vectors[1:]

# 3) Definimos la similitud del coseno "a mano" para verla por dentro
def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))          # producto escalar (A · B)
    norm_a = math.sqrt(sum(x * x for x in a))        # longitud de A (‖A‖)
    norm_b = math.sqrt(sum(y * y for y in b))        # longitud de B (‖B‖)
    return dot / (norm_a * norm_b)

# 4) Comparamos la pregunta con cada candidata y ordenamos de mayor a menor
print(f"Pregunta: {question}\n")
scores = [(cosine_similarity(q_vec, v), text) for v, text in zip(cand_vecs, candidates)]
for score, text in sorted(scores, reverse=True):
    print(f"  {score:.4f}  ->  {text}")

print("\n👉 La frase con MAYOR puntuación es la que RAG recuperaría como contexto.")


## 4️⃣ Generando embeddings con SQL (`ai_query`)

La segunda forma de generar embeddings es **directamente con SQL**, sin salir del *lakehouse*. Esto es muy potente porque permite **vectorizar tablas enteras** con una sola consulta, aprovechando el motor distribuido de Spark.

### 🪄 La función `ai_query`

`ai_query` es una **función SQL nativa de Databricks** que llama a un modelo servido desde SQL:

```sql
ai_query(<endpoint>, <entrada>)
```

| Argumento | Qué es |
|-----------|--------|
| 1º — `endpoint` | El nombre del modelo servido. Si es un **modelo de embeddings** (`databricks-gte-large-en`), devuelve un **vector** (`array<float>`). Si fuera un **LLM de chat**, devolvería **texto**. |
| 2º — `entrada` | El texto a procesar. Aquí, el texto que se convertirá en vector. |

### 🧱 La pieza clave para procesar una tabla entera

Lo normal **no** es pasar texto fijo, sino una **columna**. Para vectorizar una tabla de fragmentos harías:

```sql
SELECT
  id,
  chunk,
  ai_query('databricks-gte-large-en', chunk) AS embedding   -- ¡una fila → un vector!
FROM RAG.docs_chunks
```

> 🧩 **Python vs SQL — ¿cuál uso?**
> - **API (Python)**: ideal en *aplicaciones* y para vectorizar la **pregunta del usuario** en tiempo real.
> - **SQL (`ai_query`)**: ideal para **procesos por lotes** sobre tablas grandes dentro del lakehouse (ETL/ingesta).

> 💡 **Para la certificación**: en el flujo recomendado de Databricks **no sueles llamar a `ai_query` a mano** para vectorizar tus documentos. Al crear un *Delta Sync Index* (siguiente notebook) indicas el `embedding_model_endpoint_name` y **Databricks calcula y mantiene los embeddings por ti** automáticamente cuando la tabla cambia (gracias al *Change Data Feed*).

In [ ]:
%sql
-- Generamos el embedding (vector) de un texto directamente desde SQL.
-- Como 'databricks-gte-large-en' es un modelo de EMBEDDINGS, el resultado
-- NO es una respuesta de chat, sino un array de números (el vector).
SELECT ai_query(
    "databricks-gte-large-en",
    "Can you explain AI in ten words?"
  ) AS embedding

## ✅ Resumen y siguientes pasos

### Lo que has aprendido
- Un **embedding** es un **vector** (lista de números) que captura el **significado** de un texto.
- Textos con significados parecidos producen vectores cercanos → eso permite la **búsqueda semántica**, la base de **RAG**.
- La **similitud del coseno** mide esa cercanía (de -1 a 1) y es lo que decide qué fragmentos recupera RAG.
- En Databricks puedes generar embeddings de **dos formas**:
  - **API de OpenAI** (`client.embeddings.create`) → para aplicaciones y la pregunta del usuario.
  - **SQL** (`ai_query`) → para vectorizar tablas enteras dentro del lakehouse.
- El modelo usado fue **`databricks-gte-large-en`** (un *Foundation Model API* de **1024 dimensiones**, pago por uso).

### 🧠 Puntos típicos de certificación
| Tema | Clave |
|------|-------|
| Modelo de embeddings | `databricks-gte-large-en`, 1024 dimensiones, *pay-per-token*. |
| Consistencia | Usa el **mismo** modelo para indexar documentos y para la consulta. |
| `ai_query` | Función SQL para llamar modelos; con un modelo de embeddings devuelve un `array<float>`. |
| Secrets | Nunca pongas tokens en el código; usa *secret scopes*. |
| Automatización | El *Delta Sync Index* calcula y mantiene los embeddings automáticamente. |

### ➡️ Siguiente notebook: `Vector_Index_Creation`
Ahora que sabemos convertir texto en vectores, el siguiente paso es **guardarlos en un índice vectorial** (Mosaic AI Vector Search) para poder **buscar a gran escala** los fragmentos más parecidos a una pregunta. ¡Ahí construiremos la "R" de RAG! 🚀